# Notebook 01 — Baseline & Exploratory Data Analysis

This notebook:
- Loads the **Fraud dataset** (Seaborn built-in) as an imbalanced binary classification problem
- Performs in-depth EDA: class balance, feature distributions, correlations
- Trains a **baseline Random Forest** classifier
- Evaluates metrics and saves all artefacts for downstream notebooks

In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '.')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

from data_utils import ingest_data, preprocess_features, save_splits
from models_utils import build_and_train_model, evaluate_performance, save_model

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 5)
print('All imports OK')

## 1. Load & Inspect Data

In [ ]:
X_train, X_test, y_train, y_test = ingest_data(
    file_path='simulated:fraud',
    target_col='default',
    test_size=0.2,
    random_state=42
)

print(f'Training samples  : {len(X_train)}')
print(f'Test samples      : {len(X_test)}')
print(f'Features          : {list(X_train.columns)}')
print(f'\nClass distribution (train):')
print(y_train.value_counts())
print(f'\nImbalance ratio (majority/minority): {y_train.value_counts()[0]/y_train.value_counts()[1]:.2f}:1')

X_train.head()

## 2. Exploratory Data Analysis

In [ ]:
# --- 2a. Class imbalance visualisation ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

class_counts = y_train.value_counts()
labels = ['Not Survived', 'Survived']
colors = ['#e74c3c', '#2ecc71']

axes[0].bar(labels, class_counts.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Class Distribution — Training Set', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 2, str(v), ha='center', fontsize=11, fontweight='bold')

axes[1].pie(class_counts.values, labels=labels, autopct='%1.1f%%',
            colors=colors, startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Survival Rate — Training Set', fontsize=12, fontweight='bold')

plt.suptitle('⚠️  Class Imbalance Overview', fontsize=13)
plt.tight_layout()
os.makedirs('artifacts', exist_ok=True)
plt.savefig('artifacts/eda_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- 2b. Numeric feature distributions by target class ---
target_col = y_train.name
train_full = pd.concat([X_train, y_train], axis=1)

num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
num_cols_for_plot = num_cols[:min(6, len(num_cols))]

if num_cols_for_plot:
    fig, axes = plt.subplots(int(np.ceil(len(num_cols_for_plot)/3)), 3, figsize=(15, 8))
    axes = axes.flatten()

    for i, col in enumerate(num_cols_for_plot):
        for label, color, name in [(0, '#e74c3c', 'Class 0'), (1, '#2ecc71', 'Class 1')]:
            subset = train_full[train_full[target_col] == label][col].dropna()
            if not subset.empty:
                axes[i].hist(subset, bins=20, alpha=0.65, color=color, label=name, edgecolor='white')
        axes[i].set_title(f'Distribution of "{col}"', fontsize=11)
        axes[i].legend(fontsize=9)
        axes[i].set_xlabel(col)

    for j in range(len(num_cols_for_plot), len(axes)):
        axes[j].axis('off')
        
    plt.suptitle('Top Numeric Feature Distributions by Class', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('artifacts/eda_feature_distributions.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# --- 2c. Correlation heatmap ---
target_col = y_train.name
corr = train_full[num_cols_for_plot + [target_col]].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            linewidths=0.5, square=True, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Heatmap', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('artifacts/eda_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- 2d. Categorical composition rates ---
target_col = y_train.name
cat_cols = X_train.select_dtypes(exclude=np.number).columns.tolist()
cat_cols_for_plot = cat_cols[:min(2, len(cat_cols))]

if cat_cols_for_plot:
    fig, axes = plt.subplots(1, len(cat_cols_for_plot), figsize=(max(6, 6*len(cat_cols_for_plot)), 4))
    if len(cat_cols_for_plot) == 1: axes = [axes]
    
    for ax, col in zip(axes, cat_cols_for_plot):
        target_rate = train_full.groupby(col)[target_col].mean().sort_values(ascending=False)
        palette = sns.color_palette('Set2', len(target_rate))
        bars = ax.bar(target_rate.index.astype(str), target_rate.values, color=palette, edgecolor='white', linewidth=1.5)
        ax.set_title(f'Minority Rate by {col.capitalize()}', fontsize=11, fontweight='bold')
        ax.set_ylabel('Target Rate')
        ax.set_ylim(0, 1.1)
        for bar, val in zip(bars, target_rate.values):
            ax.text(bar.get_x() + bar.get_width()/2, val + 0.03, f'{val:.2f}',
                    ha='center', fontsize=11, fontweight='bold')

    plt.tight_layout()
    plt.savefig('artifacts/eda_categorical.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No categorical columns to visualize.")

## 3. Preprocessing

In [ ]:
X_train_proc, X_test_proc, preprocessor, feature_names = preprocess_features(X_train, X_test)

print(f'Preprocessed train shape : {X_train_proc.shape}')
print(f'Preprocessed test shape  : {X_test_proc.shape}')
print(f'Feature names:\n  {feature_names}')

## 4. Baseline Model — Random Forest

In [ ]:
baseline_model = build_and_train_model(
    X_train_proc, y_train.values,
    model_type='random_forest',
    model_params={'n_estimators': 100, 'max_depth': 8, 'class_weight': 'balanced'}
)

In [ ]:
baseline_metrics = evaluate_performance(baseline_model, X_test_proc, y_test.values, verbose=True)

In [ ]:
# --- Feature importance ---
importances = baseline_model.feature_importances_
feat_imp_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
feat_imp_df = feat_imp_df.sort_values('importance', ascending=True)

plt.figure(figsize=(9, 5))
plt.barh(feat_imp_df['feature'], feat_imp_df['importance'],
         color='#3498db', edgecolor='white')
plt.title('Random Forest — Feature Importances', fontsize=12, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('artifacts/eda_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Save Artefacts for Downstream Notebooks

In [ ]:
os.makedirs('artifacts', exist_ok=True)

# Save model
save_model(baseline_model, 'artifacts/baseline_rf_model.pkl')

# Save preprocessor
joblib.dump(preprocessor, 'artifacts/preprocessor.pkl')
print('Preprocessor saved →  artifacts/preprocessor.pkl')

# Save raw data splits (un-scaled, for generator training)
save_splits(X_train, y_train, X_test, y_test, output_dir='artifacts')

print('\n=== Baseline Metrics Summary ===')
for k, v in baseline_metrics.items():
    print(f'  {k:12s}: {v:.4f}')